In [ ]:
import pandas as pd
import joblib


In [ ]:
train_dataset = pd.read_csv("intermediate_data/train_dataset.csv")
train_dataset

In [ ]:
train_dataset.drop(columns=['Unnamed: 0', 'SK_ID_CURR', 'TARGET'], inplace=True)

In [ ]:
xgb_pipe = joblib.load("models/xgb_final_pipeline_w_50_optuna.joblib")

In [ ]:
[x for x in xgb_pipe.named_steps]

In [ ]:
booster = xgb_pipe.named_steps['xgb_model'].get_booster()
gain = booster.get_score(importance_type="gain")
total_gain = booster.get_score(importance_type="total_gain")
weight = booster.get_score(importance_type="weight")
cover = booster.get_score(importance_type="cover")

In [ ]:
col_names = xgb_pipe.named_steps['preprocess'].get_feature_names_out()
col_names = [col[col.find('__')+2:] if '__' in col else col for col in col_names]

In [ ]:
gain_df = (
    pd.DataFrame(zip(col_names, gain.values()), columns=["feature", "gain"])
).sort_values('gain', ascending=False)

total_gain_df = (
    pd.DataFrame(zip(col_names, total_gain.values()), columns=["feature", "total_gain"])
).sort_values('total_gain', ascending=False)

weight_df = (
    pd.DataFrame(zip(col_names, weight.values()), columns=["feature", "weight"])
).sort_values('weight', ascending=False)

cover_df = (
    pd.DataFrame(zip(col_names, cover.values()), columns=["feature", "cover"])
).sort_values('cover', ascending=False)


total_gain_df.head(20)


In [ ]:
import shap

In [ ]:
X_valid_trans = xgb_pipe.named_steps["preprocess"].transform(train_dataset)
feature_names = col_names

model = xgb_pipe.named_steps["xgb_model"]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_valid_trans)


In [ ]:
shap.summary_plot(shap_values, X_valid_trans, feature_names=feature_names)